# PREEMI first look at the data

In [154]:
%matplotlib inline

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

PATH = "../data/preemi-30apr2021.dta"

print(df.shape)

## Cleaning data

### Drop columns


In [155]:
df = pd.read_stata(PATH)

df[df == ""] = np.nan

columns = list(df)

# Unique columns
df = df.drop(columns=["protocolid", "personid", "pr_id", "ch_id", "mt_id"])

# Event status
df = df.drop(columns=[c for c in columns if "eventstatus_e" in c])

# Versions
df = df.drop(columns=[c for c in columns if "versionname_e" in c])

# Constant or almost unique
df = df.drop(columns=["haemounava_e1"])
df = df.drop(columns=["checklist_e1"])
df = df.drop(columns=["matstata_e2"])

# Remove columns with high NaNs
N = df.shape[0]
for col in columns:
    if col not in df: continue
    n = df[df[col].isnull()].shape[0]
    if n/N > 0.5:
        df = df.drop(columns=[col])

In [156]:
# Remove redundant _e
import collections
red_cols = collections.defaultdict(list)
for c in list(df.columns):
    x = c.split("_")
    if len(x[-1]) == 2 and x[-1][0] == "e":
        red_cols["_".join(x[:-1])] += ["_".join(x)]

def get_to_drop(v):
    b = -1
    best = None
    for x in v:
        if int(x[-1]) > b:
            b = int(x[-1])
            best = x
    return [x for x in v if x != best]

for k,v in red_cols.items():
    v = get_to_drop(v)
    df = df.drop(columns=v)

df = df.drop(columns = ["compperson_e5"])

In [157]:
# Minor cleans

df.loc[df["facid_e5"] == "FO3", "facid_e5"] = "F03"
df.loc[df["facid_e5"] == "FO2", "facid_e5"] = "F02"

In [158]:
# Remove dirty variables

df = df.drop(columns=["BWDQ_cat", "birthweightcat1", "gestagecat2", "gestagecat1"])


# Newly realised redundant
df.drop(columns = ["facid_e5"])

,studysubjectid,edd_e1,estedd_e1,mensdate_e1,matage,schlev_e1,eduyrs_e1,parity_lbsb,mathei_e1,matwei_e1,...,lbw,died,deplace,detype,matagecat,parity,gestmethod,pr_outcome,dob_day,bwdat
0,115A-03429-1,20/09/2016,1=Date of last mestrual period,13/12/2015,25.0,3=School,11.0,1.0,163.0,63.0,...,Normal birthweight,Alive,Facility delivery,Vaginal,25-29,1-3,LMP,Live birth,3.0,At delivery (between 0-6 hours)
1,12A-00001-1,02/11/2015,1=Date of last mestrual period,25/01/2015,33.0,3=School,9.0,3.0,158.0,64.0,...,Normal birthweight,Alive,Facility delivery,Vaginal,30-34,1-3,LMP,Live birth,21.0,At delivery (between 0-6 hours)
2,12A-00002-1,12/10/2015,1=Date of last mestrual period,05/01/2015,25.0,3=School,9.0,1.0,161.0,70.0,...,Normal birthweight,Alive,Facility delivery,Vaginal,25-29,1-3,LMP,Live birth,22.0,At delivery (between 0-6 hours)
3,12A-00003-1,30/10/2015,1=Date of last mestrual period,21/01/2015,23.0,3=School,12.0,0.0,155.0,49.0,...,Normal birthweight,Alive,Facility delivery,Cesarean,20-24,0,LMP,Live birth,27.0,At delivery (between 0-6 hours)
4,12A-00004-1,10/10/2015,1=Date of last mestrual period,03/01/2015,35.0,3=School,7.0,1.0,999.0,73.0,...,Normal birthweight,Alive,Facility delivery,Vaginal,≥35,1-3,LMP,Live birth,15.0,At delivery (between 0-6 hours)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11529,18A-03026-1,05/08/2017,1=Date of last mestrual period,28/10/2016,28.0,3=School,11.0,3.0,157.0,67.0,...,Normal birthweight,Alive,Facility delivery,Vaginal,25-29,1-3,LMP,Live birth,5.0,Birthweight ≥6 hours & <24 hours
11530,18A-03027-1,12/04/2017,1=Date of last mestrual period,05/07/2016,29.0,3=School,12.0,2.0,156.0,69.0,...,Normal birthweight,Alive,Facility delivery,Vaginal,25-29,1-3,LMP,Live birth,21.0,Birthweight ≥6 hours & <24 hours
11531,18A-03028-1,23/07/2017,1=Date of last mestrual period,16/10/2016,20.0,3=School,10.0,1.0,154.0,999.0,...,Normal birthweight,Alive,Facility delivery,Vaginal,20-24,1-3,LMP,Live birth,20.0,Birthweight ≥6 hours & <24 hours
11532,18A-04521-2,7/8/2016,1=Date of last mestrual period,10/1/2015,33.0,3=School,7.0,3.0,162.0,55.0,...,Low birthweight,Alive,Home delivery,Vaginal,30-34,1-3,Missing,Live birth,1.0,Birth weight missing


In [159]:
# Clean data type

def datatype_cleaner(df, col):
    df_ = df.copy()
    df_ = df_[df_[col].notnull()]
    N = df_.shape[0]
    df_["numnew"] = pd.to_numeric(df_[col], errors="coerce")
    df_["datnew"] = pd.to_datetime(df_[col], errors="coerce")
    if df_[df_["numnew"].notnull()].shape[0]/N > 0.8:
        df[col] = pd.to_numeric(df[col])
        return df
    if df_[df_["datnew"].notnull()].shape[0]/N > 0.8:
        df[col] = pd.to_datetime(df[col], errors="coerce")
        df.loc[df[col] < pd.Timestamp(1910,1,1)] = np.nan
        return df
    return df

for col in columns:
    if col not in df: continue
    df = datatype_cleaner(df, col)
    

In [160]:
df.loc[df["matage"] > 60, "matage"] = np.nan
df.loc[df["eduyrs_e1"] > 15, "eduyrs_e1"] = 15
df.loc[df["matwei_e1"] > 200, "matwei_e1"] = np.nan
df.loc[df["babwght_e2"] > 8000, "babwght_e2"] = np.nan

In [161]:
from pandas_profiling import ProfileReport

profile = ProfileReport(df, title="PREEMI exploratory analysis")
profile.to_file("PREEMI_Exploratory_Analysis.html")

Summarize dataset:   0%|          | 0/109 [00:00<?, ?it/s]

/Users/mduran/opt/anaconda3/envs/preemi/lib/python3.7/site-packages/pandas_profiling/model/summary_algorithms.py:268: RuntimeWarning: overflow encountered in long_scalars
  stats["range"] = stats["max"] - stats["min"]
/Users/mduran/opt/anaconda3/envs/preemi/lib/python3.7/site-packages/pandas_profiling/model/summary_algorithms.py:268: RuntimeWarning: overflow encountered in long_scalars
  stats["range"] = stats["max"] - stats["min"]
/Users/mduran/opt/anaconda3/envs/preemi/lib/python3.7/site-packages/missingno/missingno.py:250: UserWarning: FixedFormatter should only be used together with FixedLocator
  ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right', fontsize=fontsize)
/Users/mduran/opt/anaconda3/envs/preemi/lib/python3.7/site-packages/pandas_profiling/model/summary.py:200: UserWarning: There was an attempt to generate the bar missing values diagrams, but this failed.
    To hide this warning, disable the calculation
    (using `df.profile_report(missing_diagrams={"ba

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]